In [1]:
# ============================================================
# BLOCK 1
# Imports + basic geometry + largest connected component
# + branch graph extraction
# ============================================================

from pathlib import Path
import math
import warnings

import numpy as np
import pandas as pd
import networkx as nx

from scipy.sparse import csr_matrix, diags
from scipy.sparse.linalg import eigsh


# ------------------------------------------------------------
# Basic geometry
# ------------------------------------------------------------

def euclidean_distance(p, q):
    """
    Euclidean distance between two 3D points.
    """
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    return float(np.linalg.norm(p - q))


def edge_length(G, u, v):
    """
    Length of edge uv using the 3D positions stored in G.nodes[v]["pos"].
    """
    return euclidean_distance(G.nodes[u]["pos"], G.nodes[v]["pos"])


# ------------------------------------------------------------
# Largest connected component
# ------------------------------------------------------------

def largest_connected_component(G):
    """
    Return the largest connected component of G as a copied graph.

    The graph is relabeled to integers, but the original labels are saved
    in the node attribute "old_label".
    """
    if G.number_of_nodes() == 0:
        return G.copy()

    if G.is_directed():
        G = G.to_undirected()

    if nx.is_connected(G):
        H = G.copy()
    else:
        largest_cc = max(nx.connected_components(G), key=len)
        H = G.subgraph(largest_cc).copy()

    H = nx.convert_node_labels_to_integers(H, label_attribute="old_label")
    return H


# ------------------------------------------------------------
# Branch graph extraction
# ------------------------------------------------------------

def extract_branch_graph(G):
    """
    Extract the seed-vertex branch graph from a complete 3D karst graph.

    Seed vertices are vertices with degree != 2:
        - degree 1: extremity vertices
        - degree > 2: junction vertices

    A branch is a maximal path between two seed vertices whose internal
    vertices have degree 2.

    The returned graph is a MultiGraph because two seed vertices may be
    connected by more than one distinct branch.

    Each branch edge stores:
        euclidean_length
        curvilinear_length
        tortuosity
        path_nodes
    """

    if G.number_of_nodes() == 0:
        return nx.MultiGraph()

    if G.is_directed():
        G = G.to_undirected()

    G = G.copy()
    degrees = dict(G.degree())

    # Seed vertices: endpoints and junctions
    seeds = {v for v, deg in degrees.items() if deg != 2}

    # Special case: pure cycle, all degrees are 2.
    # Choose two artificial seed vertices to break the cycle.
    if len(seeds) == 0:
        nodes = list(G.nodes())
        if len(nodes) == 1:
            seeds = {nodes[0]}
        else:
            seeds = {nodes[0], nodes[len(nodes) // 2]}

    B = nx.MultiGraph()

    for s in seeds:
        B.add_node(
            s,
            pos=G.nodes[s]["pos"],
            old_label=G.nodes[s].get("old_label", s)
        )

    visited_directed_edges = set()

    def mark_edge(a, b):
        visited_directed_edges.add((a, b))
        visited_directed_edges.add((b, a))

    for s in seeds:
        for nbr in G.neighbors(s):

            if (s, nbr) in visited_directed_edges:
                continue

            path = [s, nbr]
            prev = s
            current = nbr
            mark_edge(s, nbr)

            # Follow the path until another seed vertex is reached.
            while current not in seeds:

                next_candidates = [x for x in G.neighbors(current) if x != prev]

                if len(next_candidates) == 0:
                    break

                nxt = next_candidates[0]

                path.append(nxt)
                mark_edge(current, nxt)

                prev, current = current, nxt

            start = path[0]
            end = path[-1]

            if start == end:
                continue

            # Curvilinear length = sum of segment lengths along the branch.
            curv_len = 0.0
            for a, b in zip(path[:-1], path[1:]):
                curv_len += edge_length(G, a, b)

            # Euclidean length = straight-line distance between branch endpoints.
            d_len = euclidean_distance(G.nodes[start]["pos"], G.nodes[end]["pos"])

            if curv_len <= 0 or d_len <= 0:
                continue

            tort = curv_len / d_len

            B.add_edge(
                start,
                end,
                euclidean_length=d_len,
                curvilinear_length=curv_len,
                tortuosity=tort,
                path_nodes=path
            )

    return B

In [2]:
# ============================================================
# BLOCK 2
# Weighted adjacency/Laplacian + spectral metric functions
# ============================================================

def weighted_adjacency_from_branch_graph(B, weight_type="euclidean"):
    """
    Build a weighted adjacency matrix from the branch graph.

    weight_type="euclidean":
        weight = 1 / d_ij

    weight_type="curvilinear":
        weight = 1 / l_ij
    """

    node_list = list(B.nodes())
    idx = {v: i for i, v in enumerate(node_list)}
    n = len(node_list)

    rows, cols, data = [], [], []

    for u, v, attr in B.edges(data=True):

        if weight_type == "euclidean":
            length = attr["euclidean_length"]
        elif weight_type == "curvilinear":
            length = attr["curvilinear_length"]
        else:
            raise ValueError("weight_type must be 'euclidean' or 'curvilinear'.")

        if length <= 0:
            continue

        w = 1.0 / length

        i = idx[u]
        j = idx[v]

        rows.extend([i, j])
        cols.extend([j, i])
        data.extend([w, w])

    A = csr_matrix((data, (rows, cols)), shape=(n, n))

    # If multiple branches connect the same seed vertices, sum the weights.
    A.sum_duplicates()

    return A, node_list


def weighted_laplacian(A):
    """
    Return weighted Laplacian L = D - A.
    """
    degrees = np.asarray(A.sum(axis=1)).ravel()
    D = diags(degrees)
    return D - A


def safe_lambda2_laplacian(L, dense_threshold=1500):
    """
    Compute lambda_2 of a symmetric Laplacian.
    """
    n = L.shape[0]

    if n <= 1:
        return np.nan

    if n <= dense_threshold:
        eigs = np.linalg.eigvalsh(L.toarray())
        eigs[np.abs(eigs) < 1e-12] = 0.0
        eigs = np.sort(eigs)
        return float(eigs[1])

    try:
        vals = eigsh(L, k=2, which="SM", return_eigenvectors=False, tol=1e-6)
        vals = np.sort(vals)
        vals[np.abs(vals) < 1e-10] = 0.0
        return float(vals[1])
    except Exception as e:
        warnings.warn(f"lambda2 computation failed: {e}")
        return np.nan


def safe_spectral_radius(A, dense_threshold=1500):
    """
    Compute spectral radius of a symmetric nonnegative weighted adjacency matrix.
    """
    n = A.shape[0]

    if n == 0:
        return np.nan

    if n == 1:
        return 0.0

    if n <= dense_threshold:
        eigs = np.linalg.eigvalsh(A.toarray())
        return float(np.max(np.abs(eigs)))

    try:
        val = eigsh(A, k=1, which="LA", return_eigenvectors=False, tol=1e-6)[0]
        return float(abs(val))
    except Exception as e:
        warnings.warn(f"spectral radius computation failed with sparse method: {e}")
        try:
            eigs = np.linalg.eigvalsh(A.toarray())
            return float(np.max(np.abs(eigs)))
        except Exception:
            return np.nan


def full_spectrum_metrics(A, L, max_dense=6000):
    """
    Compute full-spectrum metrics when the branch graph is not too large.

    Metrics:
        graph_energy
        laplacian_energy
        estrada_index
        kirchhoff_index
        log10_spanning_trees

    If branch_n > max_dense, returns NaN for these metrics.
    """
    n = A.shape[0]

    out = {
        "graph_energy": np.nan,
        "laplacian_energy": np.nan,
        "estrada_index": np.nan,
        "kirchhoff_index": np.nan,
        "log10_spanning_trees": np.nan,
    }

    if n <= 1:
        return out

    if n > max_dense:
        return out

    try:
        A_dense = A.toarray()
        L_dense = L.toarray()

        adj_eigs = np.linalg.eigvalsh(A_dense)
        lap_eigs = np.linalg.eigvalsh(L_dense)

        lap_eigs[np.abs(lap_eigs) < 1e-12] = 0.0
        positive_lap = lap_eigs[lap_eigs > 1e-12]

        weighted_total_edge_weight = A.sum() / 2.0
        weighted_avg_degree = 2.0 * weighted_total_edge_weight / n

        out["graph_energy"] = float(np.sum(np.abs(adj_eigs)))
        out["laplacian_energy"] = float(np.sum(np.abs(lap_eigs - weighted_avg_degree)))
        out["estrada_index"] = float(np.sum(np.exp(adj_eigs)))

        if len(positive_lap) == n - 1:
            out["kirchhoff_index"] = float(n * np.sum(1.0 / positive_lap))

            # Weighted Matrix-Tree Theorem:
            # tau_w = (1/n) product_{k=2}^n mu_k
            log_tau = np.sum(np.log(positive_lap)) - np.log(n)
            out["log10_spanning_trees"] = float(log_tau / np.log(10))

        return out

    except Exception as e:
        warnings.warn(f"full spectrum metrics failed: {e}")
        return out


def heat_trace_exact(L, t_values=(0.01, 0.1, 1.0, 10.0), max_dense=6000):
    """
    Exact heat trace:
        H(t) = Tr(exp(-tL)) = sum_k exp(-t mu_k)

    Computed only when branch_n <= max_dense.
    """
    n = L.shape[0]
    result = {}

    for t in t_values:
        result[f"heat_trace_t_{t:g}"] = np.nan

    if n == 0 or n > max_dense:
        return result

    try:
        eigs = np.linalg.eigvalsh(L.toarray())
        eigs[np.abs(eigs) < 1e-12] = 0.0

        for t in t_values:
            result[f"heat_trace_t_{t:g}"] = float(np.sum(np.exp(-t * eigs)))

        return result

    except Exception as e:
        warnings.warn(f"heat trace computation failed: {e}")
        return result


def safe_ratio(a, b):
    """
    Safe ratio a / b.
    """
    if a is None or b is None:
        return np.nan
    if np.isnan(a) or np.isnan(b) or b == 0:
        return np.nan
    return float(a / b)


def safe_log_ratio(a, b):
    """
    Safe logarithmic ratio log(a / b).
    """
    if a is None or b is None:
        return np.nan
    if np.isnan(a) or np.isnan(b) or a <= 0 or b <= 0:
        return np.nan
    return float(np.log(a / b))


def geometric_spectral_metrics_for_graph(
    G,
    cavename="unknown",
    max_dense=6000,
    heat_t_values=(0.01, 0.1, 1.0, 10.0),
):
    """
    Compute geometric spectral metrics for one 3D karst complete graph.

    The matrices are built on the branch graph:
        A_d, A_l, L_d, L_l.
    """

    # 1. Largest connected component
    H = largest_connected_component(G)

    # 2. Extract branch graph
    B = extract_branch_graph(H)

    # 3. Build A_d and A_l
    A_d, nodes_d = weighted_adjacency_from_branch_graph(B, weight_type="euclidean")
    A_l, nodes_l = weighted_adjacency_from_branch_graph(B, weight_type="curvilinear")

    # 4. Laplacians
    L_d = weighted_laplacian(A_d)
    L_l = weighted_laplacian(A_l)

    # 5. Branch statistics
    branch_euclidean_lengths = []
    branch_curvilinear_lengths = []
    branch_tortuosities = []

    for _, _, attr in B.edges(data=True):
        branch_euclidean_lengths.append(attr["euclidean_length"])
        branch_curvilinear_lengths.append(attr["curvilinear_length"])
        branch_tortuosities.append(attr["tortuosity"])

    branch_euclidean_lengths = np.asarray(branch_euclidean_lengths, dtype=float)
    branch_curvilinear_lengths = np.asarray(branch_curvilinear_lengths, dtype=float)
    branch_tortuosities = np.asarray(branch_tortuosities, dtype=float)

    result = {
        "cavename": cavename,

        # Largest connected component of complete graph
        "complete_lcc_n": H.number_of_nodes(),
        "complete_lcc_m": H.number_of_edges(),

        # Branch graph
        "branch_n": B.number_of_nodes(),
        "branch_m": B.number_of_edges(),

        # Branch geometry
        "total_euclidean_branch_length": (
            float(np.sum(branch_euclidean_lengths))
            if len(branch_euclidean_lengths) else np.nan
        ),
        "total_curvilinear_branch_length": (
            float(np.sum(branch_curvilinear_lengths))
            if len(branch_curvilinear_lengths) else np.nan
        ),
        "mean_branch_euclidean_length": (
            float(np.mean(branch_euclidean_lengths))
            if len(branch_euclidean_lengths) else np.nan
        ),
        "mean_branch_curvilinear_length": (
            float(np.mean(branch_curvilinear_lengths))
            if len(branch_curvilinear_lengths) else np.nan
        ),
        "mean_branch_tortuosity": (
            float(np.mean(branch_tortuosities))
            if len(branch_tortuosities) else np.nan
        ),
        "max_branch_tortuosity": (
            float(np.max(branch_tortuosities))
            if len(branch_tortuosities) else np.nan
        ),
    }

    # 6. Scalable spectral metrics
    result["rho_A_d"] = safe_spectral_radius(A_d)
    result["rho_A_l"] = safe_spectral_radius(A_l)

    result["lambda2_L_d"] = safe_lambda2_laplacian(L_d)
    result["lambda2_L_l"] = safe_lambda2_laplacian(L_l)

    # 7. Full-spectrum metrics for A_d and L_d
    d_metrics = full_spectrum_metrics(A_d, L_d, max_dense=max_dense)
    for key, value in d_metrics.items():
        result[f"{key}_d"] = value

    # 8. Full-spectrum metrics for A_l and L_l
    l_metrics = full_spectrum_metrics(A_l, L_l, max_dense=max_dense)
    for key, value in l_metrics.items():
        result[f"{key}_l"] = value

    # 9. Heat traces
    heat_d = heat_trace_exact(L_d, t_values=heat_t_values, max_dense=max_dense)
    for key, value in heat_d.items():
        result[f"{key}_d"] = value

    heat_l = heat_trace_exact(L_l, t_values=heat_t_values, max_dense=max_dense)
    for key, value in heat_l.items():
        result[f"{key}_l"] = value

    # 10. Comparison metrics
    result["rho_ratio_d_over_l"] = safe_ratio(
        result["rho_A_d"],
        result["rho_A_l"]
    )

    result["spectral_tortuosity_lambda2_ratio"] = safe_ratio(
        result["lambda2_L_d"],
        result["lambda2_L_l"]
    )

    result["kirchhoff_ratio_l_over_d"] = safe_ratio(
        result["kirchhoff_index_l"],
        result["kirchhoff_index_d"]
    )

    result["energy_log_ratio_d_over_l"] = safe_log_ratio(
        result["graph_energy_d"],
        result["graph_energy_l"]
    )

    result["laplacian_energy_log_ratio_d_over_l"] = safe_log_ratio(
        result["laplacian_energy_d"],
        result["laplacian_energy_l"]
    )

    result["estrada_log_ratio_d_over_l"] = safe_log_ratio(
        result["estrada_index_d"],
        result["estrada_index_l"]
    )

    if (
        not np.isnan(result["log10_spanning_trees_d"])
        and not np.isnan(result["log10_spanning_trees_l"])
    ):
        result["delta_log10_spanning_trees_d_minus_l"] = (
            result["log10_spanning_trees_d"]
            - result["log10_spanning_trees_l"]
        )
    else:
        result["delta_log10_spanning_trees_d_minus_l"] = np.nan

    return result, B

In [3]:
# ============================================================
# BLOCK 3
# Correct CSV loader + recursive batch processor
# ============================================================

def load_karst_graph_from_csv(edge_csv, node_pos_csv):
    """
    Load a complete 3D karst graph from two CSV files.

    This version handles:
      - semicolon-separated files
      - UTF-8 BOM characters such as '\\ufefffrom_id'
      - KNdata-public columns: from_id, to_id, node_id, x, y, z
    """

    def clean_col(c):
        return str(c).replace("\ufeff", "").strip()

    # sep=None detects comma/semicolon/tab.
    # encoding="utf-8-sig" helps remove BOM.
    edges = pd.read_csv(edge_csv, sep=None, engine="python", encoding="utf-8-sig")
    pos = pd.read_csv(node_pos_csv, sep=None, engine="python", encoding="utf-8-sig")

    edges.columns = [clean_col(c) for c in edges.columns]
    pos.columns = [clean_col(c) for c in pos.columns]

    possible_edge_cols = [
        ("from_id", "to_id"),
        ("source", "target"),
        ("u", "v"),
        ("node1", "node2"),
        ("from", "to"),
        ("i", "j"),
    ]

    edge_cols = None
    for a, b in possible_edge_cols:
        if a in edges.columns and b in edges.columns:
            edge_cols = (a, b)
            break

    if edge_cols is None:
        raise ValueError(
            f"Could not detect edge columns in {edge_csv}. "
            f"Columns are: {list(edges.columns)}"
        )

    possible_node_cols = [
        "node_id",
        "id",
        "node",
        "vertex",
        "station",
    ]

    node_col = None
    for c in possible_node_cols:
        if c in pos.columns:
            node_col = c
            break

    if node_col is None:
        raise ValueError(
            f"Could not detect node-id column in {node_pos_csv}. "
            f"Columns are: {list(pos.columns)}"
        )

    coord_options = [
        ("x", "y", "z"),
        ("X", "Y", "Z"),
        ("pos_x", "pos_y", "pos_z"),
        ("easting", "northing", "elevation"),
    ]

    coord_cols = None
    for xcol, ycol, zcol in coord_options:
        if xcol in pos.columns and ycol in pos.columns and zcol in pos.columns:
            coord_cols = (xcol, ycol, zcol)
            break

    if coord_cols is None:
        raise ValueError(
            f"Could not detect x,y,z coordinate columns in {node_pos_csv}. "
            f"Columns are: {list(pos.columns)}"
        )

    a, b = edge_cols
    xcol, ycol, zcol = coord_cols

    G = nx.Graph()

    # Add nodes with 3D coordinates
    for _, row in pos.iterrows():
        node = row[node_col]
        x = float(row[xcol])
        y = float(row[ycol])
        z = float(row[zcol])
        G.add_node(node, pos=(x, y, z))

    # Add edges
    for _, row in edges.iterrows():
        u = row[a]
        v = row[b]

        if u in G.nodes and v in G.nodes and u != v:
            G.add_edge(u, v)

    return G


def find_edge_and_position_csvs(cave_folder):
    """
    Recursively find edge CSV and node-position CSV inside one cave folder.

    Works for KNdata-public folders such as:
        clean_graph_csv/..._edges.csv
        clean_graph_csv/..._node_pos.csv
    """

    cave_folder = Path(cave_folder)
    all_csvs = list(cave_folder.rglob("*.csv"))

    edge_candidates = []
    pos_candidates = []

    for f in all_csvs:
        name = f.name.lower()

        if name.endswith("_edges.csv") or "edges" in name:
            edge_candidates.append(f)

        if name.endswith("_node_pos.csv") or "node_pos" in name:
            pos_candidates.append(f)

    return edge_candidates, pos_candidates


def process_many_karst_csv_folders_geometric(
    base_folder,
    output_csv="karst_geometric_spectral_summary.csv",
    skipped_csv="karst_geometric_spectral_skipped.csv",
    max_complete_lcc_vertices=10**9,
    max_dense=6000,
):
    """
    Process all 3D karst networks in KNdata-public style folders.

    It searches recursively inside each cave folder.
    """

    base_folder = Path(base_folder)

    results = []
    skipped = []

    cave_folders = [p for p in sorted(base_folder.iterdir()) if p.is_dir()]

    print(f"Found {len(cave_folders)} cave folders.")

    for cave_folder in cave_folders:
        cavename = cave_folder.name

        edge_candidates, pos_candidates = find_edge_and_position_csvs(cave_folder)

        if len(edge_candidates) == 0 or len(pos_candidates) == 0:
            skipped.append({
                "cavename": cavename,
                "reason": "Missing edge CSV or node-position CSV",
                "num_csv_files_found": len(list(cave_folder.rglob("*.csv"))),
            })
            print(f"Skipping {cavename}: missing edge or node-position CSV.")
            continue

        edge_csv = edge_candidates[0]
        node_pos_csv = pos_candidates[0]

        print(f"\nProcessing {cavename}")
        print(f"  edge file: {edge_csv.relative_to(cave_folder)}")
        print(f"  pos file:  {node_pos_csv.relative_to(cave_folder)}")

        try:
            G = load_karst_graph_from_csv(edge_csv, node_pos_csv)

            if G.number_of_nodes() == 0:
                skipped.append({
                    "cavename": cavename,
                    "reason": "Empty graph",
                })
                print("  Skipped: empty graph.")
                continue

            H = largest_connected_component(G)
            lcc_n = H.number_of_nodes()
            lcc_m = H.number_of_edges()

            print(f"  Largest connected component: {lcc_n} vertices, {lcc_m} edges")

            if lcc_n > max_complete_lcc_vertices:
                skipped.append({
                    "cavename": cavename,
                    "reason": f"LCC has {lcc_n} vertices > {max_complete_lcc_vertices}",
                    "complete_lcc_n": lcc_n,
                    "complete_lcc_m": lcc_m,
                })
                print("  Skipped: LCC too large.")
                continue

            metrics, B = geometric_spectral_metrics_for_graph(
                G,
                cavename=cavename,
                max_dense=max_dense,
            )

            metrics["edge_csv"] = str(edge_csv)
            metrics["node_pos_csv"] = str(node_pos_csv)
            metrics["complete_lcc_vertex_limit"] = max_complete_lcc_vertices
            metrics["full_spectrum_dense_limit_branch_n"] = max_dense

            results.append(metrics)

            print(f"  Branch graph: {metrics['branch_n']} seed vertices, {metrics['branch_m']} branches")
            print(f"  lambda2_L_d = {metrics['lambda2_L_d']}")
            print(f"  lambda2_L_l = {metrics['lambda2_L_l']}")
            print(f"  spectral tortuosity ratio = {metrics['spectral_tortuosity_lambda2_ratio']}")

        except Exception as e:
            skipped.append({
                "cavename": cavename,
                "reason": f"Error: {e}",
            })
            print(f"  ERROR for {cavename}: {e}")

    df_results = pd.DataFrame(results)
    df_skipped = pd.DataFrame(skipped)

    df_results.to_csv(output_csv, index=False)
    df_skipped.to_csv(skipped_csv, index=False)

    print("\nDone.")
    print(f"Computed metrics shape: {df_results.shape}")
    print(f"Skipped networks shape: {df_skipped.shape}")
    print(f"Saved computed metrics to: {output_csv}")
    print(f"Saved skipped networks to: {skipped_csv}")

    return df_results, df_skipped

In [4]:
# ============================================================
# BLOCK 4
# Run the analysis
# ============================================================

base_folder = r"C:\Users\roz31\Documents\GitHub\KNdata-public\caves_fulldatasets\data"

df_geom, df_skipped = process_many_karst_csv_folders_geometric(
    base_folder=base_folder,
    output_csv="karst_geometric_spectral_summary.csv",
    skipped_csv="karst_geometric_spectral_skipped.csv",
    max_complete_lcc_vertices=10**9,  # practically no vertex limit
    max_dense=6000                    # full-spectrum metrics only if branch_n <= 6000
)

print("Computed metrics shape:", df_geom.shape)
print("Skipped networks shape:", df_skipped.shape)

display(df_geom.head())
display(df_skipped.head())

# Useful quick check
important_cols = [
    "cavename",
    "complete_lcc_n",
    "complete_lcc_m",
    "branch_n",
    "branch_m",
    "rho_A_d",
    "rho_A_l",
    "lambda2_L_d",
    "lambda2_L_l",
    "spectral_tortuosity_lambda2_ratio",
    "mean_branch_tortuosity",
]

existing_cols = [c for c in important_cols if c in df_geom.columns]

display(df_geom[existing_cols].head(20))

# Save final copies
df_geom.to_csv("karst_geometric_spectral_summary_final.csv", index=False)
df_skipped.to_csv("karst_geometric_spectral_skipped_final.csv", index=False)

print("Saved final CSV files.")

Found 79 cave folders.

Processing 001_GouffreDejaVu_000
  edge file: clean_graph_csv\001_GouffreDejaVu_000_edges.csv
  pos file:  clean_graph_csv\001_GouffreDejaVu_000_node_pos.csv
  Largest connected component: 33 vertices, 32 edges
  Branch graph: 4 seed vertices, 3 branches
  lambda2_L_d = 0.02214401618724255
  lambda2_L_l = 0.012132197062415282
  spectral tortuosity ratio = 1.8252272093274184

Processing 002_Migovec_000
  edge file: clean_graph_csv\002_Migovec_000_edges.csv
  pos file:  clean_graph_csv\002_Migovec_000_node_pos.csv
  Largest connected component: 7163 vertices, 7206 edges
  Branch graph: 889 seed vertices, 927 branches
  lambda2_L_d = 2.6776971029030266e-06
  lambda2_L_l = 1.9528886917830393e-06
  spectral tortuosity ratio = 1.3711468114745535

Processing 003_Criou_000
  edge file: clean_graph_csv\003_Criou_000_edges.csv
  pos file:  clean_graph_csv\003_Criou_000_node_pos.csv
  Largest connected component: 3721 vertices, 3741 edges
  Branch graph: 436 seed vertices,

C:\Users\roz31\AppData\Local\Temp\ipykernel_12936\2115747267.py:82: UserWarning: lambda2 computation failed: ARPACK error -1: No convergence (48311 iterations, 0/2 eigenvectors converged)
  warnings.warn(f"lambda2 computation failed: {e}")


  Branch graph: 4831 seed vertices, 5288 branches
  lambda2_L_d = nan
  lambda2_L_l = nan
  spectral tortuosity ratio = nan

Processing 020_Vanoise_000
  edge file: clean_graph_csv\020_Vanoise_000_edges.csv
  pos file:  clean_graph_csv\020_Vanoise_000_node_pos.csv
  Largest connected component: 471 vertices, 484 edges
  Branch graph: 83 seed vertices, 96 branches
  lambda2_L_d = 0.00029440879352381764
  lambda2_L_l = 0.00022640289962406394
  spectral tortuosity ratio = 1.3003755429487682

Processing 021_Glacier_000
  edge file: clean_graph_csv\021_Glacier_000_edges.csv
  pos file:  clean_graph_csv\021_Glacier_000_node_pos.csv
  Largest connected component: 900 vertices, 939 edges
  Branch graph: 195 seed vertices, 234 branches
  lambda2_L_d = 0.0003882793737131849
  lambda2_L_l = 0.00032328998990313075
  spectral tortuosity ratio = 1.2010250420358741

Processing 022_CombeBryon_000
  edge file: clean_graph_csv\022_CombeBryon_000_edges.csv
  pos file:  clean_graph_csv\022_CombeBryon_000_

,cavename,complete_lcc_n,complete_lcc_m,branch_n,branch_m,total_euclidean_branch_length,total_curvilinear_branch_length,mean_branch_euclidean_length,mean_branch_curvilinear_length,mean_branch_tortuosity,...,spectral_tortuosity_lambda2_ratio,kirchhoff_ratio_l_over_d,energy_log_ratio_d_over_l,laplacian_energy_log_ratio_d_over_l,estrada_log_ratio_d_over_l,delta_log10_spanning_trees_d_minus_l,edge_csv,node_pos_csv,complete_lcc_vertex_limit,full_spectrum_dense_limit_branch_n
0,001_GouffreDejaVu_000,33,32,4,3,93.387409,150.678847,31.129136,50.226282,1.393286,...,1.825227,1.613481,0.025681,0.020935,0.000163,0.383427,C:\Users\roz31\Documents\GitHub\KNdata-public\...,C:\Users\roz31\Documents\GitHub\KNdata-public\...,1000000000,6000
1,002_Migovec_000,7163,7206,889,927,32733.589628,45964.660530,35.311316,49.584316,1.350572,...,1.371147,1.381348,0.114469,0.100989,0.002614,96.500369,C:\Users\roz31\Documents\GitHub\KNdata-public\...,C:\Users\roz31\Documents\GitHub\KNdata-public\...,1000000000,6000
2,003_Criou_000,3721,3741,436,455,17251.083994,23246.463007,37.914470,51.091127,1.235715,...,1.362257,1.351957,0.040361,0.029098,0.000464,33.153893,C:\Users\roz31\Documents\GitHub\KNdata-public\...,C:\Users\roz31\Documents\GitHub\KNdata-public\...,1000000000,6000
3,004_Matienzo_000,6058,6128,1457,1524,32123.498491,40913.598984,21.078411,26.846194,1.190334,...,1.220025,1.229286,0.026427,0.013393,0.000347,85.524777,C:\Users\roz31\Documents\GitHub\KNdata-public\...,C:\Users\roz31\Documents\GitHub\KNdata-public\...,1000000000,6000
4,005_Sakany_000,1426,1467,308,347,4202.794174,5597.633764,12.111799,16.131509,1.310812,...,1.324397,1.292436,0.035312,0.012686,0.000042,27.793620,C:\Users\roz31\Documents\GitHub\KNdata-public\...,C:\Users\roz31\Documents\GitHub\KNdata-public\...,1000000000,6000


""


,cavename,complete_lcc_n,complete_lcc_m,branch_n,branch_m,rho_A_d,rho_A_l,lambda2_L_d,lambda2_L_l,spectral_tortuosity_lambda2_ratio,mean_branch_tortuosity
0,001_GouffreDejaVu_000,33,32,4,3,0.114163,0.111268,0.022144,0.012132,1.825227,1.393286
1,002_Migovec_000,7163,7206,889,927,0.985753,0.985526,0.000003,0.000002,1.371147,1.350572
2,003_Criou_000,3721,3741,436,455,16.440799,16.440335,0.000008,0.000006,1.362257,1.235715
3,004_Matienzo_000,6058,6128,1457,1524,9.711119,9.711039,0.000002,0.000001,1.220025,1.190334
4,005_Sakany_000,1426,1467,308,347,70.710830,70.710788,0.000276,0.000208,1.324397,1.310812
5,006_ReveEveille_000,78,77,4,3,0.141584,0.141194,0.006691,0.004724,1.416319,1.178394
6,007_Tsanfleuron_000,206,206,11,11,0.117176,0.097371,0.003049,0.002214,1.377209,1.293380
7,008_UltimaPatagonia_000,605,618,105,118,0.683943,0.681842,0.000540,0.000461,1.170734,1.358814
8,009_Folly_000,4709,4746,641,678,1.164796,1.123229,0.000007,0.000005,1.321396,1.247180
9,010_PlaninaPoljana_000,1600,1619,309,326,1.832212,1.831532,0.000033,0.000026,1.304173,1.283570


Saved final CSV files.
